In [ ]:
# Linalg libraries
import numpy as np

# Plotting libraries
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns

# Data management libraries
import h5py as hf
from dataclasses import dataclass
import glob
import imageio
from matplotlib.colors import LogNorm
from rich.progress import track

In [ ]:
@dataclass
class ModelTrajectory:
    # Model data
    width: int
    depth: int
    activation: str
    lr: float
    architecture: str
    ds_size: float

    # Loss data
    train_loss: np.ndarray
    
    train_loss_std: np.ndarray
    test_loss: np.ndarray
    test_loss_std: np.ndarray

    # Accuracy data
    train_acc: np.ndarray
    train_acc_std: np.ndarray
    test_acc: np.ndarray
    test_acc_std: np.ndarray

    # CV data
    entropy: np.ndarray
    entropy_std: np.ndarray
    trace: np.ndarray
    trace_std: np.ndarray

In [ ]:
def load_model_trajectory(file_root, file_signature, accuracy = False):
    # Glob all files with the correct signature
    files = glob.glob(file_root + '/*' + file_signature + '*.h5')

    # Extract model parameters
    model_params = files[0].split("/")[-1].split('_')
    width = int(model_params[2])
    depth = int(model_params[3])
    activation = model_params[4]
    ds_size = float(model_params[5])    
    lr = float(model_params[7])
    architecture = model_params[1]

    # Load data
    train_losses = []
    test_losses = []

    if accuracy:
        train_accuracies = []
        test_accuracies = []
    
    entropies = []
    traces = []

    for file in files:
        with hf.File(file, 'r') as f:
            if file.split("/")[-1].split("_")[0] == "train":
                train_losses.append(f['loss'][:])
                train_loss = f["loss"][:]
                if accuracy:
                    train_accuracies.append(f['accuracy'][:])
                    train_accuracy = f['accuracy'][:]
                entropies.append(f['entropy'][:])
                traces.append(f['trace'][:])
            else:
                test_losses.append(f['loss'][:])
                test_loss = f['loss'][:]
                if accuracy:
                    test_accuracies.append(f['accuracy'][:])
                    test_accuracy = f['accuracy'][:]

    # Take the means, std and build the ModelTrajectory object
    # train_loss = train_losses[0] # np.nanmean(train_losses, axis=0)
    # test_loss = test_losses[0] # np.nanmean(test_losses, axis=0)
    train_loss_std = None # np.nanstd(train_losses, axis=0)
    test_loss_std = None # np.nanstd(test_losses, axis=0)

    if accuracy:
        # train_accuracy = np.mean(train_accuracies, axis=0)
        # test_accuracy = np.mean(test_accuracies, axis=0)

        train_accuracy_std = None # np.std(train_accuracies, axis=0)
        test_accuracy_std = None #np.std(test_accuracies, axis=0)

    entropy = entropies[-1] # np.nanmean(entropies, axis=0)
    trace = traces[-1] # np.nanmean(traces, axis=0)
    entropy_std = None # np.std(entropies, axis=0)
    trace_std = None # np.std(traces, axis=0)

    return ModelTrajectory(
        width=width,
        depth=depth,
        activation=activation,
        lr=lr,
        architecture=architecture,
        ds_size=ds_size,
        train_loss=train_loss,
        train_loss_std=train_loss_std,
        test_loss=test_loss,
        test_loss_std=test_loss_std,
        entropy=entropy,
        entropy_std=entropy_std,
        trace=trace,
        trace_std=trace_std,
        train_acc=train_accuracy,
        train_acc_std=train_accuracy_std,
        test_acc=test_accuracy,
        test_acc_std=test_accuracy_std
    )

def load_model_trajectories(root_path):
    files = glob.glob(root_path + '/*.h5')

    # Get unique names
    unique_names = []
    for file in files:
        unique_names.append(
            "_".join(file.split('/')[-1].split('_')[2:7])
            )

    unique = np.unique(unique_names)

    # Load data
    data = []
    for name in unique:
        data.append(load_model_trajectory(root_path, name, accuracy=True))

    return data   

In [ ]:
data = load_model_trajectories("./scale-data/")

In [ ]:
tanh_stats = {}
relu_stats = {}

t_entropies = {"min": [], "max": []}
t_losses = {"min": [], "max": []}
t_traces = {"min": [], "max": []}

r_entropies = {"min": [], "max": []}
r_losses = {"min": [], "max": []}
r_traces = {"min": [], "max": []}


for item in data:
    if item.activation == "tanh":
        t_entropies["min"].append(np.min(item.entropy))
        t_entropies["max"].append(np.max(item.entropy))

        t_losses["min"].append(np.min(item.train_loss))
        t_losses["max"].append(np.max(item.train_loss))

        t_traces["min"].append(np.min(item.trace))
        t_traces["max"].append(np.max(item.trace))

    if item.activation == "relu":
        r_entropies["min"].append(np.min(item.entropy))
        r_entropies["max"].append(np.max(item.entropy))

        r_losses["min"].append(np.min(item.train_loss))
        r_losses["max"].append(np.max(item.train_loss))

        r_traces["min"].append(np.min(item.trace))
        r_traces["max"].append(np.max(item.trace))

tanh_stats = {
    "entropy": [np.min(t_entropies["min"]), np.max(t_entropies["max"])],
    "trace": [np.min(t_traces["min"]), np.max(t_traces["max"])],
    "loss": [np.min(t_losses["min"]), np.max(t_losses["max"])]
}

relu_stats = {
    "entropy": [np.min(r_entropies["min"]), np.max(r_entropies["max"])],
    "trace": [np.min(r_traces["min"]), np.max(r_traces["max"])],
    "loss": [np.min(r_losses["min"]) + 1e-10, np.max(r_losses["max"])]
}

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(15, 15))

t_entropies = []
t_losses = []
t_traces = []

t_widths = []
t_depths = []

r_entropies = []
r_losses = []
r_traces = []

r_widths = []
r_depths = []

for item in data:
    if item.activation == "tanh":
        lindex = np.argmin(item.test_loss)
        index = 0

        t_entropies.append(item.entropy[index])
        t_losses.append(item.test_loss[lindex])
        t_traces.append(item.trace[index])

        t_widths.append(item.width)
        t_depths.append(item.depth)

    if item.activation == "relu":
        
        lindex = np.argmin(item.test_loss)
        index = 0
        r_entropies.append(item.entropy[index])
        r_losses.append(item.test_loss[lindex])
        r_traces.append(item.trace[index])
        
        r_widths.append(item.width)
        r_depths.append(item.depth)

# ReLU plots
sc1 = ax[0][0].scatter(
    r_widths, 
    r_depths, 
    c=r_entropies, 
    s=100, 
    # norm=LogNorm(),
    # vmin=relu_stats["entropy"][0], 
    # vmax=relu_stats["entropy"][1],
    cmap="coolwarm_r"
)
plt.colorbar(sc1, ax=ax[0][0], orientation='vertical', label="Entropy")

sc2 = ax[1][0].scatter(
    r_widths, 
    r_depths, 
    c=r_traces, 
    s=100, 
    norm=LogNorm(),
    cmap="coolwarm"
)
plt.colorbar(sc2, ax=ax[1][0], orientation='vertical', label="Trace")

sc3 = ax[2][0].scatter(
    r_widths, 
    r_depths, 
    c=r_losses, 
    s=100, 
    norm=LogNorm(),
    cmap="coolwarm"
)
plt.colorbar(sc3, ax=ax[2][0], orientation='vertical', label="Loss")

# tanH plots
sc4 = ax[0][1].scatter(
    t_widths, 
    t_depths, 
    c=t_entropies, 
    s=100, 
    # norm=LogNorm(),
    # vmin=min(r_entropies), 
    # vmax=max(r_entropies),
    cmap="coolwarm_r"
)
plt.colorbar(sc4, ax=ax[0][1], orientation='vertical', label="Entropy")

sc5 = ax[1][1].scatter(
    t_widths, 
    t_depths, 
    c=t_traces, 
    s=100, 
    norm=LogNorm(),
    cmap="coolwarm"
)
plt.colorbar(sc5, ax=ax[1][1], orientation='vertical', label="Trace")

sc6 = ax[2][1].scatter(
    t_widths, 
    t_depths, 
    c=t_losses, 
    s=100,
    norm=LogNorm(),
    cmap="coolwarm"

)
plt.colorbar(sc6, ax=ax[2][1], orientation='vertical', label="Loss")

for i in range(3):
    for j in range(2):
        ax[i][j].set_xscale("log")

ax[0][0].set_title("ReLU")
ax[0][1].set_title("TanH")
# plt.savefig("test-loss-mid.png")
plt.show()

In [ ]:
for i in range(100):

    fig, ax = plt.subplots(3, 2, figsize=(15, 15))

    t_entropies = []
    t_losses = []
    t_traces = []

    t_widths = []
    t_depths = []

    r_entropies = []
    r_losses = []
    r_traces = []

    r_widths = []
    r_depths = []

    for item in data:
        if item.activation == "tanh":
            index = i

            t_entropies.append(item.entropy[index])
            t_losses.append(item.train_loss[index])
            t_traces.append(item.trace[index])

            t_widths.append(item.width)
            t_depths.append(item.depth)

        if item.activation == "relu":
            
            index = i
            r_entropies.append(item.entropy[index])
            r_losses.append(item.train_loss[index])
            r_traces.append(item.trace[index])
            
            r_widths.append(item.width)
            r_depths.append(item.depth)

    widths = list(np.unique(np.logspace(1, 3.5, 20, dtype=int)))
    depths=[1, 2, 3, 5, 6, 7, 8, 9, 10]

    depth_map = {depths[i]: i for i in range(len(depths))}
    width_map = {widths[i]: i for i in range(len(widths))}

    # ReLU plots
    entropy_grid = np.zeros((len(depths), len(widths)))
    trace_grid = np.zeros((len(depths), len(widths)))
    loss_grid = np.zeros((len(depths), len(widths)))
    for width, depth, entropy, trace, loss in zip(r_widths, r_depths, r_entropies, r_traces, r_losses):
        entropy_grid[depth_map[depth], width_map[width]] = entropy
        trace_grid[depth_map[depth], width_map[width]] = trace
        loss_grid[depth_map[depth], width_map[width]] = loss

    im = ax[0][0].pcolormesh(widths, depths, entropy_grid, cmap="coolwarm_r", shading="gouraud", vmin=relu_stats["entropy"][0], vmax=relu_stats["entropy"][1])
    plt.colorbar(im, ax=ax[0][0], orientation='vertical', label="Entropy")
    im = ax[1][0].pcolormesh(widths, depths, trace_grid, cmap="coolwarm", norm=LogNorm(vmin=relu_stats["trace"][0], vmax=relu_stats["trace"][1]), shading="gouraud")
    plt.colorbar(im, ax=ax[1][0], orientation='vertical', label="Trace")
    im = ax[2][0].pcolormesh(widths, depths, loss_grid, cmap="coolwarm", norm=LogNorm(vmin=relu_stats["loss"][0], vmax=relu_stats["loss"][1]), shading="gouraud")
    plt.colorbar(im, ax=ax[2][0], orientation='vertical', label="Loss")

    # tanH plots
    entropy_grid = np.zeros((len(depths), len(widths)))
    trace_grid = np.zeros((len(depths), len(widths)))
    loss_grid = np.zeros((len(depths), len(widths)))
    for width, depth, entropy, trace, loss in zip(t_widths, t_depths, t_entropies, t_traces, t_losses):
        entropy_grid[depth_map[depth], width_map[width]] = entropy
        trace_grid[depth_map[depth], width_map[width]] = trace
        loss_grid[depth_map[depth], width_map[width]] = loss

    im = ax[0][1].pcolormesh(widths, depths, entropy_grid, cmap="coolwarm_r", shading="gouraud", vmin=tanh_stats["entropy"][0], vmax=tanh_stats["entropy"][1])
    plt.colorbar(im, ax=ax[0][1], orientation='vertical', label="Entropy")
    im = ax[1][1].pcolormesh(widths, depths, trace_grid, cmap="coolwarm", norm=LogNorm(vmin=tanh_stats["trace"][0], vmax=tanh_stats["trace"][1]), shading="gouraud")
    plt.colorbar(im, ax=ax[1][1], orientation='vertical', label="Trace")
    im = ax[2][1].pcolormesh(widths, depths, loss_grid, cmap="coolwarm", norm=LogNorm(vmin=tanh_stats["loss"][0], vmax=tanh_stats["loss"][1]), shading="gouraud")
    plt.colorbar(im, ax=ax[2][1], orientation='vertical', label="Loss")

    # Set labels
    for p in range(3):
        for j in range(2):
            ax[p][j].set_ylabel("Depth")
            ax[p][j].set_xlabel("Width")

    # Set scales
    ax[0][1].set_xscale("log")
    ax[1][1].set_xscale("log")
    ax[2][1].set_xscale("log")
    ax[0][0].set_xscale("log")
    ax[1][0].set_xscale("log")
    ax[2][0].set_xscale("log")

    # Set titles
    ax[0][0].set_title("ReLU")
    ax[0][1].set_title("TanH")
    plt.savefig(f"frame_{i}.png")
    plt.close()

In [ ]:
filenames = [f"frame_{i}.png" for i in range(100)]

# Create GIF
with imageio.get_writer('train-loss.gif', mode='I', duration=5.0) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

In [ ]:
filenames = []

for i in track(range(100)):
    fig, ax = plt.subplots(3, 3, figsize=(15, 15))

    t_entropies = []
    t_losses = []
    t_traces = []

    t_widths = []
    t_depths = []

    r_entropies = []
    r_losses = []
    r_traces = []

    r_widths = []
    r_depths = []

    for item in data:
        if item.activation == "tanh":
            t_entropies.append(item.entropy[i])
            t_losses.append(item.train_loss[i])
            t_traces.append(item.trace[i])

            t_widths.append(item.width)
            t_depths.append(item.depth)

        if item.activation == "relu":
            r_entropies.append(item.entropy[i])
            r_losses.append(item.train_loss[i])
            r_traces.append(item.trace[i])
            
            r_widths.append(item.width)
            r_depths.append(item.depth)

    # ReLU plots
    sc1 = ax[0][0].scatter(
        r_widths, 
        r_depths, 
        c=r_entropies, 
        s=100, 
        vmin=relu_stats["entropy"][0], 
        vmax=relu_stats["entropy"][1]
    )
    plt.colorbar(sc1, ax=ax[0][0], orientation='vertical', label="Entropy")

    sc2 = ax[1][0].scatter(
        r_widths, 
        r_depths, 
        c=r_traces, 
        s=100, 
        norm=LogNorm(relu_stats["trace"][0], relu_stats["trace"][1])
    )
    plt.colorbar(sc2, ax=ax[1][0], orientation='vertical', label="Trace")

    sc3 = ax[2][0].scatter(
        r_widths, 
        r_depths, 
        c=r_losses, 
        s=100, 
        norm=LogNorm(relu_stats["loss"][0] + 1e-10, relu_stats["loss"][1])
    )
    plt.colorbar(sc3, ax=ax[2][0], orientation='vertical', label="Loss")

    

    # tanH plots
    sc4 = ax[0][1].scatter(
        t_widths, 
        t_depths, 
        c=t_entropies, 
        s=100, 
        vmin=tanh_stats["entropy"][0], 
        vmax=tanh_stats["entropy"][1]
    )
    plt.colorbar(sc4, ax=ax[0][1], orientation='vertical', label="Entropy")

    sc5 = ax[1][1].scatter(
        t_widths, 
        t_depths, 
        c=t_traces, 
        s=100, 
        norm=LogNorm(vmin=tanh_stats["trace"][0], vmax=tanh_stats["trace"][1])
    )
    plt.colorbar(sc5, ax=ax[1][1], orientation='vertical', label="Trace")

    sc6 = ax[2][1].scatter(
        t_widths, 
        t_depths, 
        c=t_losses, 
        s=100, 
        norm=LogNorm(vmin=tanh_stats["loss"][0], vmax=tanh_stats["loss"][1])
    )
    plt.colorbar(sc6, ax=ax[2][1], orientation='vertical', label="Loss")

    for k in range(3):
        for m in range(2):
            ax[k][m].set_xscale("log")

    ax[0][0].set_title(f"ReLU Epoch: {i}")
    ax[0][1].set_title(f"TanH Epoch: {i}")
    
    plt.savefig(f"frame_{i}.png")
    filenames.append(f"frame_{i}.png")
    plt.close()

In [ ]:
filenames = [f"frame_{i}.png" for i in range(100)]

In [ ]:
# Create GIF
with imageio.get_writer('test-loss-movie.gif', mode='I', duration=5.0) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

In [ ]:
for i in range(100):
    fig, ax = plt.subplots(3, 2, figsize=(15, 15))
    t_entropies = []
    t_losses = []
    t_traces = []
    traces = []
    losses = []
    entropies = []
    t_widths = []
    t_depths = []

    for item in data:
        if item.activation == "tanh":
            t_entropies.append(item.entropy[i])
            t_losses.append(item.train_loss[i])
            t_traces.append(item.trace[i])

            t_widths.append(item.width)
            t_depths.append(item.depth)
            losses.append(item.train_loss[:100])
            entropies.append(item.entropy[:100])
            traces.append(item.trace[:100])

    en_plot = ax[0][0].scatter(
        t_widths, 
        t_depths, 
        c=t_entropies, 
        s=100, 
        vmin=tanh_stats["entropy"][0], 
        vmax=tanh_stats["entropy"][1]
    )
    plt.colorbar(en_plot, ax=ax[0][1], orientation='vertical', label="Entropy")
    ax[0][0].set_xscale("log")
    ax[0][1].plot(np.mean(entropies, axis=0))
    # ax[0][1].fill_between(
    #     np.arange(0, 100),
    #     np.mean(entropies, axis=0) - np.std(entropies, axis=0),
    #     np.mean(entropies, axis=0) + np.std(entropies, axis=0),
    #     alpha=0.5
    # )
    ax[0][1].vlines(i, np.min(np.mean(entropies, axis=0)), np.max(np.mean(entropies, axis=0)), color='red')

    tr_plot = ax[1][0].scatter(
        t_widths, 
        t_depths, 
        c=t_traces, 
        s=100, 
        norm=LogNorm(vmin=tanh_stats["trace"][0], vmax=tanh_stats["trace"][1])
    )
    plt.colorbar(tr_plot, ax=ax[1][1], orientation='vertical', label="Trace")
    ax[1][0].set_xscale("log")
    ax[1][1].plot(np.mean(traces, axis=0))
    # ax[1][1].fill_between(
    #     np.arange(0, 100),
    #     np.mean(traces, axis=0) - np.std(traces, axis=0),
    #     np.mean(traces, axis=0) + np.std(traces, axis=0),
    #     alpha=0.5
    # )
    ax[1][1].set_yscale("log")
    ax[1][1].vlines(i, 1, np.max(np.mean(traces, axis=0)), color='red')

    l_plot = ax[2][0].scatter(
        t_widths, 
        t_depths, 
        c=t_losses, 
        s=100, 
        norm=LogNorm(vmin=tanh_stats["loss"][0], vmax=tanh_stats["loss"][1])
    )
    ax[2][0].set_xscale("log")
    plt.colorbar(l_plot, ax=ax[2][1], orientation='vertical', label="Loss")

    ax[2][1].plot(np.mean(losses, axis=0))
    # ax[2][1].fill_between(
    #     np.arange(0, 100),
    #     np.mean(losses, axis=0) - np.std(losses, axis=0),
    #     np.mean(losses, axis=0) + np.std(losses, axis=0),
    #     alpha=0.5
    # )
    ax[2][1].set_yscale("log")
    ax[2][1].vlines(i, np.min(np.mean(losses, axis=0)), np.max(np.mean(losses, axis=0)), color='red')

    plt.plot()
    plt.savefig(f"tanh_{i}.png")
    plt.close()

In [ ]:
filenames = [f"tanh_{i}.png" for i in range(100)]

In [ ]:
# Create GIF
with imageio.get_writer('train-loss-tanh.gif', mode='I', duration=5.0) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(8.0, 5.0))

# Start
t_entropies = []
t_losses = []
t_traces = []

t_widths = []
t_depths = []

r_entropies = []
r_losses = []
r_traces = []

r_widths = []
r_depths = []

for item in data:
    if item.activation == "relu":
        
        index = 0
        r_entropies.append(item.entropy[index])
        r_losses.append(item.train_loss[index])
        r_traces.append(item.trace[index])
        
        r_widths.append(item.width)
        r_depths.append(item.depth)

widths = list(np.unique(np.logspace(1, 3.5, 20, dtype=int)))
depths=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

depth_map = {depths[i]: i for i in range(len(depths))}
width_map = {widths[i]: i for i in range(len(widths))}

# ReLU plots
entropy_grid = np.zeros((len(depths), len(widths)))
trace_grid = np.zeros((len(depths), len(widths)))
loss_grid = np.zeros((len(depths), len(widths)))
for width, depth, entropy, trace, loss in zip(r_widths, r_depths, r_entropies, r_traces, r_losses):
    entropy_grid[depth_map[depth], width_map[width]] = entropy
    trace_grid[depth_map[depth], width_map[width]] = trace
    loss_grid[depth_map[depth], width_map[width]] = loss

im = ax[0][0].pcolormesh(widths, depths, entropy_grid, cmap="coolwarm", shading="gouraud", vmin=relu_stats["entropy"][0], vmax=relu_stats["entropy"][1])
plt.colorbar(im, ax=ax[0][0], orientation='vertical', label="Entropy")
im = ax[0][1].pcolormesh(widths, depths, trace_grid, cmap="coolwarm", norm=LogNorm(vmin=relu_stats["trace"][0], vmax=relu_stats["trace"][1]), shading="gouraud")
plt.colorbar(im, ax=ax[0][1], orientation='vertical', label="Trace")
im = ax[0][2].pcolormesh(widths, depths, loss_grid, cmap="coolwarm_r", norm=LogNorm(vmin=relu_stats["loss"][0], vmax=relu_stats["loss"][1]), shading="gouraud")
plt.colorbar(im, ax=ax[0][2], orientation='vertical', label="Loss")

# Final
t_entropies = []
t_losses = []
t_traces = []

t_widths = []
t_depths = []

r_entropies = []
r_losses = []
r_traces = []

r_widths = []
r_depths = []

for item in data:
    if item.activation == "relu":
        
        index = 99
        r_entropies.append(item.entropy[index])
        r_losses.append(item.train_loss[index])
        r_traces.append(item.trace[index])
        
        r_widths.append(item.width)
        r_depths.append(item.depth)

widths = list(np.unique(np.logspace(1, 3.5, 20, dtype=int)))
depths=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

depth_map = {depths[i]: i for i in range(len(depths))}
width_map = {widths[i]: i for i in range(len(widths))}

# ReLU plots
entropy_grid = np.zeros((len(depths), len(widths)))
trace_grid = np.zeros((len(depths), len(widths)))
loss_grid = np.zeros((len(depths), len(widths)))
for width, depth, entropy, trace, loss in zip(r_widths, r_depths, r_entropies, r_traces, r_losses):
    entropy_grid[depth_map[depth], width_map[width]] = entropy
    trace_grid[depth_map[depth], width_map[width]] = trace
    loss_grid[depth_map[depth], width_map[width]] = loss

im = ax[1][0].pcolormesh(widths, depths, entropy_grid, cmap="coolwarm", shading="gouraud", vmin=relu_stats["entropy"][0], vmax=relu_stats["entropy"][1])
plt.colorbar(im, ax=ax[1][0], orientation='vertical', label="Entropy")
im = ax[1][1].pcolormesh(widths, depths, trace_grid, cmap="coolwarm", norm=LogNorm(vmin=relu_stats["trace"][0], vmax=relu_stats["trace"][1]), shading="gouraud")
plt.colorbar(im, ax=ax[1][1], orientation='vertical', label="Trace")
im = ax[1][2].pcolormesh(widths, depths, loss_grid, cmap="coolwarm_r", norm=LogNorm(vmin=relu_stats["loss"][0], vmax=relu_stats["loss"][1]), shading="gouraud")
plt.colorbar(im, ax=ax[1][2], orientation='vertical', label="Loss")

# Set labels
for p in range(2):
    for j in range(3):
        ax[p][j].set_ylabel("Depth")
        ax[p][j].set_xlabel("Width")
        ax[p][j].set_xscale("log")

ax[0][0].set_title("a)", loc="left")
ax[1][0].set_title("b)", loc="left")

plt.tight_layout()
plt.savefig("phases.pdf")
plt.show()